In [ ]:
import pandas as pd
import os

# Filtering national model
This script is an optional part of the workflow that copies the inputs from the national model (or any other model) and filters the data to a set of specified nodes. For example, to create a model of Alberta, write ['AB_North', 'AB_South', 'AB_Central'] in province buses. Note that the connections to other provinces will not be represented and must be added manually as required.

In [ ]:
# ✏️⚙️
MODEL_NAME = 'national_model'
NEW_MODEL_NAME = 'Alberta'
province_buses = ['AB_North', 'AB_South', 'AB_Central']

In [3]:
path = os.path.join(os.getcwd(), 'model', MODEL_NAME)
new_path = os.path.join(os.getcwd(), 'model', NEW_MODEL_NAME)
os.makedirs(new_path, exist_ok=True)

In [4]:
def filter_components(file_name, buses, old_path, new_path):
    df = pd.read_csv(os.path.join(old_path, file_name))
    df = df[df.bus.isin(buses)]
    if not df.empty:
        df.to_csv(os.path.join(new_path, file_name), index=False)
    return df

def filter_timeseries(file_name, component_df, old_path, new_path):
    df = pd.read_csv(os.path.join(old_path, file_name), index_col=0, parse_dates=True)
    df = df.loc[:, component_df.name]
    if not df.empty:
        df.to_csv(os.path.join(new_path, file_name))
    return df

In [41]:
gens = filter_components('generators.csv', province_buses, path, new_path)
gens

,name,bus,model,p_nom,p_nom_extendable,build_year,lifetime,group
0,AB_Central_gas_CC_2025_to_2035,AB_Central,gas_CC,1346.00,False,2025,10.0,NaN
1,AB_Central_gas_CC_2025_to_2040,AB_Central,gas_CC,2809.00,False,2025,15.0,NaN
2,AB_Central_gas_CC_2025_to_2045,AB_Central,gas_CC,287.00,False,2025,20.0,NaN
3,AB_Central_gas_CC_2025_to_2050,AB_Central,gas_CC,19.00,False,2025,25.0,NaN
4,AB_Central_gas_CC_2025_to_2055,AB_Central,gas_CC,161.05,False,2025,30.0,NaN
...,...,...,...,...,...,...,...,...
448,AB_Central_OPT_wind-646,AB_Central,wind_new,0.00,True,2030,NaN,ungrouped
449,AB_North_OPT_wind-581,AB_North,wind_new,0.00,True,2030,NaN,ungrouped
450,AB_South_OPT_wind-760,AB_South,wind_new,0.00,True,2030,NaN,ungrouped
471,AB_Central_hydro_run,AB_Central,hydro_ror,16.80,False,0,NaN,ungrouped


In [42]:
renewable_gens = gens[gens.model.isin(['wind_2021', 'wind_new', 'default_solar_PV', 'hydro_ror'])]
gen_cf = filter_timeseries('generators-p_max_pu.csv', renewable_gens, path, new_path)
gen_cf.head()

,AB_South_wind_ons_654_retire_2030,AB_Central_solar_PV_668_retire_2055,AB_Central_solar_PV_670_retire_2055,AB_Central_wind_ons_670_retire_2035,AB_Central_solar_PV_671_retire_2050,AB_South_solar_PV_672_retire_2055,AB_South_solar_PV_673_retire_2055,AB_South_solar_PV_674_retire_2055,AB_South_solar_PV_675_retire_2055,AB_South_wind_ons_676_retire_2030,...,AB_South_solar_PV_806_retire_2055,AB_South_wind_ons_806_retire_2050,AB_Central_OPT_PV-780,AB_North_OPT_PV-533,AB_South_OPT_PV-676,AB_Central_OPT_wind-646,AB_North_OPT_wind-581,AB_South_OPT_wind-760,AB_Central_hydro_run,AB_South_hydro_run
2025-01-01 00:00:00,0.0855,0.0,0.0,0.0844,0.0,0.0,0.0,0.0,0.0,0.0842,...,0.0,0.0459,0.0,0.0,0.0,0.0898,0.1081,0.0946,0.0482,0.0243
2025-01-01 01:00:00,0.1323,0.0,0.0,0.1304,0.0,0.0,0.0,0.0,0.0,0.1275,...,0.0,0.0358,0.0,0.0,0.0,0.1397,0.1577,0.1378,0.0484,0.0243
2025-01-01 02:00:00,0.1859,0.0,0.0,0.1841,0.0,0.0,0.0,0.0,0.0,0.1793,...,0.0,0.0277,0.0,0.0,0.0,0.1951,0.2110,0.1857,0.0486,0.0243
2025-01-01 03:00:00,0.2291,0.0,0.0,0.2289,0.0,0.0,0.0,0.0,0.0,0.2239,...,0.0,0.0263,0.0,0.0,0.0,0.2396,0.2491,0.2242,0.0488,0.0242
2025-01-01 04:00:00,0.2666,0.0,0.0,0.2680,0.0,0.0,0.0,0.0,0.0,0.2643,...,0.0,0.0351,0.0,0.0,0.0,0.2769,0.2805,0.2616,0.0489,0.0242


In [43]:
links = pd.read_csv(os.path.join(path, 'links.csv'))
links = links[(links.bus0.isin(province_buses)) | (links.bus1.isin(province_buses))]
external_connections = set(links[~links.bus0.isin(province_buses)].bus0) | set(links[~links.bus1.isin(province_buses)].bus1)
links.to_csv(os.path.join(new_path, 'links.csv'), index=False)

In [44]:
buses = pd.read_csv(os.path.join(path, 'buses.csv'))
buses = buses[(buses.name.isin(province_buses)) | (buses.name.isin(external_connections))]
buses.to_csv(os.path.join(new_path, 'buses.csv'), index=False)
buses

,name,x,y,province
0,AB_Central,-113.459539,53.433811,AB
1,AB_North,-113.006052,56.380991,AB
2,AB_South,-113.718899,50.955789,AB
4,BC_South,-123.160951,49.790166,BC
22,SK,-104.575222,50.395902,SK


In [9]:
loads = filter_components('loads.csv', province_buses, path, new_path)
loads

,name,bus
0,AB_Central_load,AB_Central
1,AB_North_load,AB_North
2,AB_South_load,AB_South
3,AB_South_US_trade,AB_South


In [46]:
load_file = None
for dirpath, _, filenames in os.walk(path):
    for filename in filenames:
        if 'loads_forecast_' in filename:
            load_file = os.path.join(dirpath, filename).rsplit('\\', 1)[-1]
if load_file is None:
    raise FileNotFoundError("No loads_forecast file found.")

load_forecast = filter_timeseries(load_file, loads, path, new_path)
load_forecast

,AB_Central_load,AB_North_load,AB_South_load,AB_South_US_trade
2025-01-01 00:00:00,5320.60,2451.52,2844.08,-179.0
2025-01-01 01:00:00,5235.56,2412.33,2798.62,-180.0
2025-01-01 02:00:00,5212.91,2401.90,2786.52,-180.0
2025-01-01 03:00:00,5167.62,2381.03,2762.31,-180.0
2025-01-01 04:00:00,5105.23,2352.28,2728.95,-175.0
...,...,...,...,...
2050-12-31 19:00:00,8317.00,3638.52,4926.67,-185.0
2050-12-31 20:00:00,8240.62,3605.10,4881.43,-185.0
2050-12-31 21:00:00,8298.46,3630.41,4915.69,-185.0
2050-12-31 22:00:00,8436.40,3690.75,4997.40,-185.0


In [47]:
storage_units = filter_components('storage_units.csv', province_buses, path, new_path)
storage_units

,name,bus,p_nom,model,max_hours,state_of_charge_initial_per_period,state_of_charge_initial,p_nom_extendable,build_year,spill_cost,lifetime
0,AB_Central_daily,AB_Central,482.0,hydro_storage,24.00,True,8676.0,False,0,3.3,NaN
1,AB_South_daily,AB_South,299.0,hydro_storage,24.00,True,5382.0,False,0,3.3,NaN
25,AB_Central_default_liion_battery_1.75h_2025_to...,AB_Central,20.0,default_liion_battery,1.75,NaN,NaN,False,2025,NaN,20.0
26,AB_Central_default_liion_battery_1.75h_2025_to...,AB_Central,60.0,default_liion_battery,1.75,NaN,NaN,False,2025,NaN,25.0
27,AB_Central_default_liion_battery_4.0h_2030_to_...,AB_Central,110.0,default_liion_battery,4.00,NaN,NaN,False,2030,NaN,25.0
28,AB_North_default_liion_battery_1.75h_2025_to_2045,AB_North,20.0,default_liion_battery,1.75,NaN,NaN,False,2025,NaN,20.0
29,AB_North_default_liion_battery_1.75h_2025_to_2050,AB_North,80.0,default_liion_battery,1.75,NaN,NaN,False,2025,NaN,25.0
30,AB_South_default_liion_battery_2.0h_2025_to_2045,AB_South,10.0,default_liion_battery,2.00,NaN,NaN,False,2025,NaN,20.0
31,AB_South_default_liion_battery_2.0h_2030_to_2055,AB_South,80.0,default_liion_battery,2.00,NaN,NaN,False,2030,NaN,25.0
32,AB_South_default_liion_battery_4.0h_2030_to_2055,AB_South,221.1,default_liion_battery,4.00,NaN,NaN,False,2030,NaN,25.0


In [48]:
hydro_storage = storage_units[storage_units.model.isin(['hydro_storage'])]
inflows = filter_timeseries('storage_units-inflow.csv', hydro_storage, path, new_path)
inflows

,AB_Central_daily,AB_South_daily
2025-01-01 00:00:00,67.19,41.68
2025-01-01 01:00:00,67.19,41.68
2025-01-01 02:00:00,67.19,41.68
2025-01-01 03:00:00,67.19,41.68
2025-01-01 04:00:00,67.19,41.68
...,...,...
2025-12-31 19:00:00,89.03,55.23
2025-12-31 20:00:00,89.03,55.23
2025-12-31 21:00:00,89.03,55.23
2025-12-31 22:00:00,89.03,55.23


In [49]:
snapshots = pd.read_csv(os.path.join(path, 'snapshots.csv'), index_col=0, parse_dates=True)
snapshots.to_csv(os.path.join(new_path, 'snapshots.csv'))